# Assistente Virtual por Voz (Híbrido)
Speech-to-Text + Text-to-Speech + Plugins + Similaridade (TF-IDF)

Este notebook implementa um assistente por voz completo: ele escuta o usuário, interpreta a intenção e executa ações por plugins.
Fazemos isso para demonstrar uma arquitetura modular de NLP aplicada, fácil de evoluir com novos comandos e integrações.

In [12]:
%pip install speechrecognition pyttsx3==2.71 wikipedia scikit-learn nltk pyaudio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [9]:
import re
import unicodedata
import uuid
import webbrowser
import platform

import speech_recognition as sr
import pyttsx3
import wikipedia

from IPython.display import HTML, Javascript, display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk

In [13]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gusta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gusta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Speech-to-Text

Nesta etapa criamos o componente responsável por capturar áudio do microfone e converter fala em texto.
Fazemos isso porque todo o fluxo do assistente depende de transformar comandos de voz em uma entrada textual processável pelo motor de intenções.

In [14]:
class SpeechToText:
    def __init__(self, visualizer=None, timeout=8, phrase_time_limit=15, pause_threshold=1.2):
        self.recognizer = sr.Recognizer()
        self.visualizer = visualizer
        self.timeout = timeout
        self.phrase_time_limit = phrase_time_limit
        self.recognizer.pause_threshold = pause_threshold

    def listen(self):
        if self.visualizer:
            self.visualizer.set_state('listening', 'Ouvindo...')

        with sr.Microphone() as source:
            print('Ouvindo...')
            self.recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = self.recognizer.listen(
                source,
                timeout=self.timeout,
                phrase_time_limit=self.phrase_time_limit
            )

        if self.visualizer:
            self.visualizer.set_state('processing', 'Processando áudio...')

        try:
            text = self.recognizer.recognize_google(audio, language='pt-BR')
            print('Você:', text)
            return text.lower()
        except sr.WaitTimeoutError:
            return ''
        except:
            return ''

## Indicador Visual (Luz Animada por Estado)

Aqui implementamos uma interface visual que mostra em tempo real o estado do assistente (ocioso, ouvindo, processando e falando).
Fazemos isso para melhorar a experiência do usuário com feedback imediato, deixando claro o que o sistema está fazendo em cada momento.

In [16]:
class VoiceLightAnimator:
    def __init__(self):
        self.element_id = f"voice-light-{uuid.uuid4().hex}"
        display(HTML(self._build_html()))
        self.set_state('idle', 'Aguardando comando...')

    def _build_html(self):
        html = """
        <style>
            #__ID__ {
                margin: 14px auto;
                max-width: 420px;
                padding: 20px 14px;
                border-radius: 16px;
                text-align: center;
                background: radial-gradient(circle at 50% 20%, #1f2937, #111827);
                color: #e5e7eb;
                font-family: 'Segoe UI', sans-serif;
                border: 1px solid #334155;
            }

            #__ID__ .orb {
                width: 82px;
                height: 82px;
                margin: 0 auto 10px;
                border-radius: 50%;
                position: relative;
                background: #64748b;
                box-shadow: 0 0 12px #64748b;
                transition: background 220ms ease, box-shadow 220ms ease;
                animation: va-idle-pulse 2.2s infinite;
            }

            #__ID__ .status {
                font-size: 14px;
                letter-spacing: 0.2px;
                opacity: 0.95;
            }

            #__ID__.listening .orb {
                background: #22d3ee;
                box-shadow: 0 0 20px #22d3ee, 0 0 44px rgba(34, 211, 238, 0.6);
                animation: va-listening-pulse 0.9s infinite;
            }

            #__ID__.processing .orb {
                background: #f59e0b;
                box-shadow: 0 0 20px #f59e0b, 0 0 44px rgba(245, 158, 11, 0.55);
                animation: va-processing-pulse 0.55s infinite;
            }

            #__ID__.speaking .orb {
                background: #22c55e;
                box-shadow: 0 0 22px #22c55e, 0 0 48px rgba(34, 197, 94, 0.6);
                animation: va-speaking-pulse 0.35s infinite;
            }

            @keyframes va-idle-pulse {
                0%, 100% { transform: scale(1); opacity: 0.92; }
                50% { transform: scale(1.04); opacity: 1; }
            }

            @keyframes va-listening-pulse {
                0%, 100% { transform: scale(1); }
                50% { transform: scale(1.12); }
            }

            @keyframes va-processing-pulse {
                0%, 100% { transform: scale(1); }
                50% { transform: scale(1.18); }
            }

            @keyframes va-speaking-pulse {
                0%, 100% { transform: scale(1); }
                50% { transform: scale(1.24); }
            }
        </style>

        <div id=\"__ID__\" class=\"idle\">
            <div class=\"orb\"></div>
            <div class=\"status\">Aguardando comando...</div>
        </div>
        """
        return html.replace('__ID__', self.element_id)

    def set_state(self, state, message=''):
        safe_state = str(state).replace('\\', '\\\\').replace("'", "\\'")
        safe_message = str(message).replace('\\', '\\\\').replace("'", "\\'")
        js = """
        (function() {
            const root = document.getElementById('__ID__');
            if (!root) return;

            root.className = '__STATE__';

            const status = root.querySelector('.status');
            if (status && '__MESSAGE__') {
                status.textContent = '__MESSAGE__';
            }
        })();
        """
        js = js.replace('__ID__', self.element_id).replace('__STATE__', safe_state).replace('__MESSAGE__', safe_message)
        display(Javascript(js))

## Text-to-Speech

Nesta seção construímos o módulo que transforma respostas textuais em áudio, incluindo seleção de voz em português e fallback de reprodução.
Fazemos isso para fechar o ciclo conversacional do assistente, permitindo que ele responda por voz de forma natural ao usuário.

In [17]:
class TextToSpeech:
    def __init__(self, visualizer=None, rate=180):
        self.visualizer = visualizer
        self.rate = rate
        self.engine = self._create_engine()

    def _is_pt_voice(self, voice):
        chunks = []
        for attr in ('id', 'name'):
            value = getattr(voice, attr, '')
            if value:
                chunks.append(str(value).lower())

        for lang in getattr(voice, 'languages', []) or []:
            if isinstance(lang, (bytes, bytearray)):
                try:
                    lang = lang.decode('utf-8', errors='ignore')
                except Exception:
                    lang = str(lang)
            chunks.append(str(lang).lower())

        haystack = ' '.join(chunks)
        return (
            ('pt-br' in haystack)
            or ('portuguese' in haystack)
            or ('brazil' in haystack)
            or ('pt_' in haystack)
            or ('pt-' in haystack)
        )

    def _create_engine(self):
        driver = 'sapi5' if platform.system().lower() == 'windows' else None
        engine = pyttsx3.init(driverName=driver) if driver else pyttsx3.init()
        engine.setProperty('rate', self.rate)

        try:
            voices = engine.getProperty('voices') or []
            pt_voice = next((v for v in voices if self._is_pt_voice(v)), None)
            if pt_voice is not None:
                engine.setProperty('voice', pt_voice.id)
        except Exception as e:
            print(f'[TTS] Nao foi possivel selecionar voz PT no pyttsx3: {e}')

        return engine

    def _speak_with_pyttsx3(self, text):
        try:
            self.engine.say(text)
            self.engine.runAndWait()
            return True
        except Exception as e:
            print(f'[TTS] Falha no pyttsx3: {e}')
            return False

    def speak(self, text):
        if self.visualizer:
            self.visualizer.set_state('speaking', 'Falando...')

        print('Assistente:', text)
        spoken = self._speak_with_pyttsx3(text)

        if not spoken:
            print('[TTS] Nao foi possivel reproduzir audio neste ambiente.')

        if self.visualizer:
            self.visualizer.set_state('idle', 'Aguardando proximo comando...')

## Plugin Loader

Este componente gerencia o registro e a recuperação dos plugins disponíveis no assistente.
Fazemos isso para desacoplar funcionalidades específicas (como YouTube e Wikipedia) do núcleo do sistema, facilitando manutenção e expansão.

In [18]:
class PluginLoader:
    def __init__(self):
        self.plugins = []

    def register(self, plugin):
        self.plugins.append(plugin)

    def get_plugins(self):
        return self.plugins

## Intent Engine (TF-IDF)

Nesta etapa definimos o motor de intenções que normaliza o texto, vetorização com TF-IDF e calcula similaridade para escolher a melhor intenção.
Fazemos isso para interpretar comandos de linguagem natural de forma robusta, mesmo quando o usuário fala variações das frases de treino.

In [19]:
class IntentEngine:
    def __init__(self, plugins):
        self.plugins = plugins
        self._train()

    def _normalize_text(self, text):
        text = text.lower().strip()
        text = unicodedata.normalize('NFD', text)
        text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _train(self):
        self.phrases = []
        self.mapping = []

        for plugin in self.plugins:
            for intent, examples in plugin.get_intents().items():
                for ex in examples:
                    self.phrases.append(self._normalize_text(ex))
                    self.mapping.append((plugin, intent))

        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.X = self.vectorizer.fit_transform(self.phrases)

    def predict(self, text):
        clean_text = self._normalize_text(text)
        vec = self.vectorizer.transform([clean_text])
        sims = cosine_similarity(vec, self.X)
        idx = sims.argmax()
        plugin, intent = self.mapping[idx]
        return plugin, intent, sims[0][idx]

## Plugins

Aqui implementamos os plugins de ação, cada um com intenções, extração de parâmetros e execução da tarefa correspondente.
Fazemos isso para separar a lógica de cada domínio e permitir adicionar novos recursos ao assistente sem alterar o motor principal.

In [20]:
class YouTubePlugin:
    def get_intents(self):
        return {
            'youtube_search': [
                'buscar video no youtube',
                'assistir video',
                'procurar video no youtube'
            ],
            'open_youtube': [
                'abrir youtube'
            ]
        }

    def extract(self, text):
        match = re.search(r'(sobre|de)\s(.+)', text)
        query = match.group(2) if match else text
        return {'query': query}

    def run(self, data):
        if data['intent'] == 'youtube_search':
            query = data['query'].replace(' ', '+')
            query = query.replace('no youtube', '').replace('video', '').strip()
            url = f'https://www.youtube.com/results?search_query={query}'
            webbrowser.open(url)
            return f"Buscando {data['query']} no YouTube"
        webbrowser.open('https://youtube.com')
        return 'Abrindo YouTube'


class WikipediaPlugin:
    def get_intents(self):
        return {
            'wikipedia_search': [
                'pesquisar na wikipedia',
                'buscar na wikipedia',
                'quero saber sobre',
                'procurar por na wikipedia',
                'abrir wikipedia',
                'wikipedia'
            ]
        }

    def extract(self, text):
        match = re.search(r'(sobre|de)\s(.+)', text)
        query = match.group(2) if match else text
        return {'query': query}

    def run(self, data):
        wikipedia.set_lang('pt')
        try:
            query = data['query'].lower()
            query = re.sub(r'\bwikip[eé]dia\b', '', query)
            query = re.sub(r'\b(pesquisar|buscar|procurar|sobre|na|no|por)\b', ' ', query)
            query = re.sub(r'\s+', ' ', query).strip()

            print(f'Query extraída para Wikipedia: "{query}"')

            if not query:
                return 'Diga um tema para pesquisar na Wikipedia.'

            return wikipedia.summary(query, sentences=2)
        except:
            return 'Não encontrei na Wikipedia'

## Execução

Nesta seção inicializamos os componentes, registramos plugins e executamos o loop principal de interação por voz.
Fazemos isso para integrar todas as partes criadas anteriormente e validar o funcionamento completo do assistente de ponta a ponta.

In [22]:
visualizer = VoiceLightAnimator()

stt = SpeechToText(visualizer, timeout=10, phrase_time_limit=20, pause_threshold=1.3)
tts = TextToSpeech(visualizer)

loader = PluginLoader()
loader.register(YouTubePlugin())
loader.register(WikipediaPlugin())

engine = IntentEngine(loader.get_plugins())

tts.speak('Assistente iniciado')

while True:
    text = stt.listen()

    if not text:
        visualizer.set_state('idle', 'Não consegui entender. Tente novamente...')
        continue

    if 'sair' in text:
        tts.speak('Encerrando')
        break

    visualizer.set_state('processing', 'Interpretando intenção...')
    plugin, intent, score = engine.predict(text)

    if score < 0.2:
        tts.speak('Não entendi bem')
        continue

    data = plugin.extract(text)
    data['intent'] = intent

    visualizer.set_state('processing', 'Executando ação...')
    response = plugin.run(data)
    tts.speak(response)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Assistente: Assistente iniciado


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Você: sair


<IPython.core.display.Javascript object>

Assistente: Encerrando


<IPython.core.display.Javascript object>

## Outros testes

In [33]:
engine.predict('procurar por PNL na wikipédia')

(<__main__.WikipediaPlugin at 0x2647db81d30>,
 'wikipedia_search',
 np.float64(0.9001649738935876))

In [37]:
wk = WikipediaPlugin()
data = wk.extract('procurar por PNL na wikipedia')
print(data)
wk.run(data)

{'query': 'procurar por PNL na wikipedia'}
Query extraída para Wikipedia: "pnl"


'Programação neurolinguística (PNL) é uma abordagem pseudocientífica que visa aproximar comunicação, desenvolvimento pessoal e psicoterapia criada por Richard Bandler e John Grinder na California, Estados Unidos, na década de 1970.\nOs criadores da PNL afirmam que existe uma conexão entre os processos neurológicos ("neuro-"), a linguagem (linguística) e os padrões comportamentais aprendidos através da experiência (programação), e que estes podem ser alterados para alcançar informações específicas e metas na vida.'

In [11]:
tts = TextToSpeech()
tts.speak("Teste de voz do assistente")

Assistente: Teste de voz do assistente
